# Problem Set · Case 7 · Where should the Gold tier start? (Part II, individual)
**BUS-G 492 · Causal decisions with AI**

In [ ]:
# --- Setup: run this cell first -----------------------------------------------
import os
DATA_URL = "https://raw.githubusercontent.com/boyoung-seo/492-coding/main/cases/case7_rdd_gold/data/"
for candidate in ["data/", "cases/case7_rdd_gold/data/", "../cases/case7_rdd_gold/data/", "../../../cases/case7_rdd_gold/data/", "../../../../cases/case7_rdd_gold/data/"]:
    if os.path.exists(candidate + "customers.csv"):
        DATA = candidate
        break
else:
    DATA = DATA_URL
print("Reading data from:", DATA)

## Something you didn't know

> **From:** Ben Kowalczyk, Loyalty Operations
> **Re:** how Gold works, exactly
>
> Gold isn't something customers apply for. It's **automatic**: anyone whose spending in a calendar year is at least **\$1,000.00** gets Gold for the next year. No exceptions. (Rules: `gold_rules.csv`.)
>
> One quirk: every December, some customers who are just short of \$1,000 make a **small extra purchase** to qualify.

**Use of AI:** strictly prohibited for questions with a * sign: **Q4, Q5, Q6, Q9**. Permitted for Q7 and Q8.

## Why Part I went wrong

Answer Q4–Q6 **before writing any code**, in your own words.

**Q4\* (2 points).** Why does the Loyalty Operations email change how you read your Part I results? Who becomes Gold, and what were Gold members like in 2025, before they had Gold?

**Q4 answer** (double-click to type)



**Q5\* (3 points).** Think about a customer who spent \$990 in 2025 and one who spent \$1,010. How different are they likely to be, apart from Gold status? Why does that make the comparison right at \$1,000 useful?

**Q5 answer** (double-click to type)



**Q6\* (2 points).** What could the December top-up purchases do to a comparison right at \$1,000? How could you check for them in the data, and how could you limit the damage?

**Q6 answer** (double-click to type)

- **What the top-ups could do:** 
- **How to check for them, and how to limit the damage:** 

## Redo the analysis

You may use AI for Q7 and Q8. This time **tell it the whole story**: paste the Loyalty Operations email and say that the data has one row per customer, and that `gold_2026` is 1 exactly when `spend_2025` is at least \$1,000.

### Load the data

This is a new notebook, so nothing from Part 1 is loaded yet. Run this cell to load the Part 1 data again, then the next cell for the new files.

In [ ]:
import statsmodels.formula.api as smf
import pandas as pd, numpy as np
df = pd.read_csv(DATA + "customers.csv")
print(df.shape)
df.head()

In [ ]:
rules = pd.read_csv(DATA + "gold_rules.csv"); print(rules.rule[0])

### Step 1 · Picture it

Run the code below and look. It puts customers with 2025 spending between \$500 and \$1,500 into **\$25 bins** and plots average 2026 spending per bin. Gold starts at \$1,000 (dashed line). Nothing to report: is there a jump at \$1,000, and how big is it compared with the Part I gap?

In [ ]:
import matplotlib.pyplot as plt
near = df[(df.spend_2025 >= 500) & (df.spend_2025 < 1500)].copy()
near["bin"] = near.spend_2025 // 25 * 25 + 12.5          # bin midpoint
ax = near.groupby("bin").spend_2026.mean().plot(marker="o", ls="", figsize=(8, 4))
ax.axvline(1000, color="gray", ls="--", lw=1)
ax.set_xlabel("2025 spending ($25 bins)"); ax.set_ylabel("average 2026 spending ($)");


## Q7 (3 points) · Look back

### Step 2 · Look back: estimate the jump (regression discontinuity) → ETT at the cutoff

Steps 2–3 use the **past data**: who became Gold in 2026 (from their 2025 spending) and what they spent in 2026. They estimate what Gold did for members near the \$1,000 line: the effect on the treated (ETT) at the cutoff. Nothing about 2027 yet.

Let x = 2025 spending − \$1,000. The code below keeps customers within a **bandwidth** of the cutoff and regresses 2026 spending on `gold_2026`, x and `gold_2026 × x` (a separate line on each side). The coefficient on `gold_2026` is the jump. The intercept is predicted 2026 spending for a non-Gold customer right at \$1,000: spending just below the cutoff. Run it and report the jump with the ±\$200 bandwidth.

In [ ]:
df["x"] = df.spend_2025 - 1000
for bw in [100, 200, 300]:
    d = df[df.x.abs() <= bw]
    m = smf.ols("spend_2026 ~ gold_2026 * x", d).fit()
    print(f"±${bw}: jump ${m.params['gold_2026']:.0f} (se {m.bse['gold_2026']:.0f}), "
          f"spending just below the cutoff ${m.params['Intercept']:,.0f}, {len(d):,} customers")


### Step 3 · Check for gaming (bunching and a donut)

The code below counts customers per **\$10 bin** of 2025 spending around \$1,000, then re-estimates Step 2 **leaving out** customers within \$25 of the cutoff (a "donut"). Run it and compare the donut jumps with Step 2.

In [ ]:
print("Customers per $10 bin of 2025 spending:")
print((df.spend_2025 // 10 * 10).astype(int).value_counts().sort_index().loc[960:1030].to_string())
for bw in [100, 200, 300]:
    d = df[(df.x.abs() <= bw) & (df.x.abs() >= 25)]       # donut: drop customers within $25 of the cutoff
    m = smf.ols("spend_2026 ~ gold_2026 * x", d).fit()
    print(f"donut ±${bw}: jump ${m.params['gold_2026']:.0f} (se {m.bse['gold_2026']:.0f}), "
          f"spending just below the cutoff ${m.params['Intercept']:,.0f}, {len(d):,} customers")


**Q7 answer** (double-click to type)

- **Jump in 2026 spending at \$1,000, with a ±\$200 bandwidth (\$):** 
- **Is there a pile-up just above \$1,000? What happens to the jump when you drop customers within \$25 of the cutoff, and which estimate do you trust more? Why?** 

## Q8 (2 points) · Look ahead: predict 2027 profit → choose T

Now **predict** what a new threshold would earn in 2027: apply the effect from Steps 2–3 to the customers your threshold would make Gold, based on their 2026 spending. This is a forecast: your deployment will show the actual 2027 result.

1. Express the jump as a **% of spending just below the cutoff** (the jump divided by the intercept). Use the estimate you trust more.
2. Harbor & Pine keeps a **35%** margin on extra spending and pays **\$120** a year per Gold member, so Gold pays for itself for a customer when 0.35 × (jump %) × spending is more than \$120.
3. Choose T. Predicted profit = Σ (0.35 × jump % × 2026 spending − \$120) over customers with 2026 spending ≥ T in your sample, × 10 for all 600,000 customers.

In [ ]:
# Your code here


**Q8 answer** (double-click to type)

- **Jump as a % of spending just below the cutoff (and which estimate you used):** 
- **Your T (\$):** 
- **Predicted 2027 profit for all 600,000 customers (\$):** 

## Q9\* (3 points) · Recommendation

Write your recommendation to Harbor & Pine's Director of Loyalty (five to eight sentences): the threshold T you recommend and its predicted 2027 profit; why the comparison in Part I was misleading; and the key assumption your answer depends on.

**Q9 answer** (double-click to type)



---
**Before you submit:** choose Runtime → Restart session and run all, so every cell runs and shows its output. Then File → Download → Download .ipynb, named Lastname_Firstname_case7_part2.ipynb.

**Submit one zip file** containing two documents: your own Part II ipynb and your AI transcript with a signed statement.

- **AI transcript with a signed statement:** the complete conversation from every AI tool you used, as a share link or a full export (not screenshots or excerpts), and a signed statement that it is complete. If you used no AI tool, submit only the signed statement saying so. The AI submission is not graded and only used for assessing learning.